# Récupération des fichiers les plus volumineux pour analyse images

In [2]:
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)

In [7]:
files = fs.find("s3://mateomorin/legifrance/documents/2017", detail=True)

In [12]:
import pandas as pd

file_sizes = []

for path, info in files.items():
    if info["type"] == "file" and info["size"] >= 1e6:
        file_sizes.append({"file": path, "size_mo": round(info["size"]/(1e6), 2)})

df_sizes = pd.DataFrame(file_sizes)

In [17]:
df_sizes.sort_values("size_mo", ascending=False).head(10)["file"].to_list()

['mateomorin/legifrance/documents/2017/11/ACCOTEXT000036508884.docx',
 'mateomorin/legifrance/documents/2017/10/ACCOTEXT000036676416.docx',
 'mateomorin/legifrance/documents/2017/11/ACCOTEXT000036595833.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000036595804.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037151099.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037170501.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000036745268.docx',
 'mateomorin/legifrance/documents/2017/11/ACCOTEXT000036508797.docx',
 'mateomorin/legifrance/documents/2017/09/ACCOTEXT000036723314.docx',
 'mateomorin/legifrance/documents/2017/12/ACCOTEXT000037908855.docx']

In [ ]:
import os

from dotenv import load_dotenv
from markitdown import MarkItDown
from openai import OpenAI

load_dotenv(override=True)

llm_client = OpenAI(
    base_url=os.environ["LLM_API_URL"],
    api_key=os.environ["LLM_API_KEY"],
)

md = MarkItDown(
    enable_plugins=True,
    llm_client=llm_client,
    llm_model="gemma4-26b-moe",
    llm_prompt="Ecris tout le texte que tu vois sur l'image, en préservant la structure des paragraphes et des titres. Veille à respecter l'ortographe, la syntaxe et la police de caractère (gras, itallique, ...). TOUT DOIT ÊTRE ECRIT EN FORMAT MARKDOWN",
)

In [5]:
result = md.convert("ACCOTEXT000036508884.pdf")

In [22]:
basic_md = MarkItDown(
    enable_plugins=False,
)

In [10]:
import docx

In [16]:
doc = docx.Document("ACCOTEXT000036508884.docx")

In [18]:
count = 0

for par in doc.paragraphs:
    if 'graphicData' in par._p.xml or 'imagedata' in par._p.xml:
        count += 1

print(count)

42


In [24]:
basic_md.convert("../example_data/ACCOTEXT000036508884.docx").markdown

'![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-emf;base64...)\n\n![](data:image/x-em

# Conversion pdf

In [1]:
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)
metadata = pd.read_parquet("s3://mateomorin/legifrance/metadata/acco_metadata_2018.parquet", filesystem=fs)

In [51]:
with fs.open("s3://mateomorin/legifrance/documents/2018/01/ACCOTEXT000036906348.docx", "rb") as f:
    content = f.read()

In [54]:
import tempfile
import subprocess

paths= [
    "s3://mateomorin/legifrance/documents/2018/01/ACCOTEXT000036906348.docx",
    "s3://mateomorin/legifrance/documents/2018/09/ACCOTEXT000037636403.docx",
    "s3://mateomorin/legifrance/documents/2018/04/ACCOTEXT000037000089.docx"
]

for doc_path in paths:
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_docx = os.path.join(tmp_dir, "input.docx")
        fs.get(doc_path, local_docx)

        cmd = ['libreoffice', '--headless', '--convert-to', 'pdf', '--outdir', tmp_dir, local_docx]
        subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        local_pdf = os.path.join(tmp_dir, "input.pdf")
        print(local_pdf)

/tmp/tmpmtnzh5wp/input.pdf
/tmp/tmpgmfp1p6p/input.pdf
/tmp/tmpaygxdg1y/input.pdf


In [66]:
import os
import subprocess
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmConvertOptions,
    VlmPipelineOptions,
)
from docling.datamodel.vlm_engine_options import ApiVlmEngineOptions, VlmEngineType
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline


def docx_body_has_images_fast(docx_path: str) -> bool:
    """
    Vérifie en quelques millisecondes via zipfile si le corps du document
    contient au moins une image, en ignorant strictement les headers et footers.
    """
    try:
        with zipfile.ZipFile(docx_path, "r") as z:
            namelist = z.namelist()

            # 1. Vérifier si des fichiers de relations existent pour le document principal
            rels_path = "word/_rels/document.xml.rels"
            doc_path = "word/document.xml"

            if rels_path not in namelist or doc_path not in namelist:
                return False

            # 2. Récupérer les rId des relations de type "image"
            rels_xml = z.read(rels_path)
            rels_root = ET.fromstring(rels_xml)

            image_rids = set()
            for rel in rels_root:
                rel_type = rel.attrib.get("Type", "")
                # Type standard OpenXML pour une image
                if rel_type.endswith("/image"):
                    rel_id = rel.attrib.get("Id")
                    if rel_id:
                        image_rids.add(rel_id)

            if not image_rids:
                return False

            # 3. Vérifier si un de ces rId est utilisé dans word/document.xml
            doc_xml = z.read(doc_path).decode("utf-8", errors="ignore")
            
            # Recherche directe de la présence de r:embed="rIdX" ou r:link="rIdX"
            for rid in image_rids:
                if f'="{rid}"' in doc_xml or f'="{rid}"' in doc_xml:
                    return True

            return False

    except Exception as e:
        print(f"Erreur lors de l'inspection rapide du DOCX : {e}")
        return False


def convert_docx_to_pdf_with_libreoffice(docx_path: str) -> str:
    """Convertit un fichier DOCX en PDF via LibreOffice CLI."""
    output_dir = Path(docx_path).parent
    cmd = [
        "libreoffice",
        "--headless",
        "--convert-to",
        "pdf",
        docx_path,
        "--outdir",
        str(output_dir),
    ]
    subprocess.run(cmd, check=True)
    return str(output_dir / f"{Path(docx_path).stem}.pdf")


input_docx = "../example_data/ACCOTEXT000036508884.docx"

# 1. Détection instantanée
needs_ocr = docx_body_has_images_fast(input_docx)

if needs_ocr:
    print("Image(s) détectée(s) dans le corps du DOCX : conversion PDF + VLM.")
    pdf_path = convert_docx_to_pdf_with_libreoffice(input_docx)

    engine_options = ApiVlmEngineOptions(
        runtime_type=VlmEngineType.API,
        url=f"{os.environ['LLM_API_URL'].rstrip('/')}/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"},
        params={
            "model": "gemma4-26b-moe",
            "temperature": 0.0,
        },
        timeout=120,
    )

    vlm_options = VlmConvertOptions.from_preset(
        "gemma_27b",
        engine_options=engine_options,
    )

    pipeline_options = VlmPipelineOptions(
        vlm_options=vlm_options,
        enable_remote_services=True,
    )

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=VlmPipeline,
                pipeline_options=pipeline_options,
            )
        }
    )

    result = doc_converter.convert(pdf_path)

    if os.path.exists(pdf_path):
        os.remove(pdf_path)

else:
    print("Aucune image dans le corps : conversion directe DOCX via Docling.")
    doc_converter = DocumentConverter()
    result = doc_converter.convert(input_docx)

# 2. Export Markdown final
markdown_output = result.document.export_to_markdown()
print(markdown_output)

Image(s) détectée(s) dans le corps du DOCX : conversion PDF + VLM.
convert /home/onyxia/work/scrapping_accords/example_data/ACCOTEXT000036508884.docx as a Writer document -> /home/onyxia/work/scrapping_accords/example_data/ACCOTEXT000036508884.pdf using filter : writer_pdf_Export
Overwriting: /home/onyxia/work/scrapping_accords/example_data/ACCOTEXT000036508884.pdf
E.Leclerc

ACCORD GESTION PREVISIONNELLE DES EMPLOIS ET COMPETENCES *17 Novembre 2017*

E.Leclerc L

# Préambule

**ENTRE LES SOUSSIGNÉS :**

**La Société SCA OUEST** , Société Anonyme à capital variable dont le siège social est sis 1 Route de Cordemais 44360 SAINT-ETIENNE-DE-MONTLUC, inscrite au Registre du Commerce de Saint-Nazaire, sous le numéro B 007080021.

Représentée par XXXXXXX agissant en qualité de Directeur, **D'UNE PART**

**ET**

XXXXXXXX, délégué syndical désigné respectivement par l' organisation syndicale CGT, **D'AUTRE PART**

**IL A ETE CONVENU CE QUI SUIT :**

2

# E.Leclerc

# Sommaire

1. Orientations s

In [ ]:
import os
import subprocess
import xml.etree.ElementTree as ET

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmConvertOptions,
    VlmPipelineOptions,
)
from docling.datamodel.vlm_engine_options import ApiVlmEngineOptions, VlmEngineType
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline


def docx_body_has_images_fast(docx_path: str) -> bool:
    """
    Vérifie en quelques millisecondes via zipfile si le corps du document
    contient au moins une image, en ignorant strictement les headers et footers.
    """
    try:
        with zipfile.ZipFile(docx_path, "r") as z:
            namelist = z.namelist()

            # 1. Vérifier si des fichiers de relations existent pour le document principal
            rels_path = "word/_rels/document.xml.rels"
            doc_path = "word/document.xml"

            if rels_path not in namelist or doc_path not in namelist:
                return False

            # 2. Récupérer les rId des relations de type "image"
            rels_xml = z.read(rels_path)
            rels_root = ET.fromstring(rels_xml)

            image_rids = set()
            for rel in rels_root:
                rel_type = rel.attrib.get("Type", "")
                # Type standard OpenXML pour une image
                if rel_type.endswith("/image"):
                    rel_id = rel.attrib.get("Id")
                    if rel_id:
                        image_rids.add(rel_id)

            if not image_rids:
                return False

            # 3. Vérifier si un de ces rId est utilisé dans word/document.xml
            doc_xml = z.read(doc_path).decode("utf-8", errors="ignore")
            
            # Recherche directe de la présence de r:embed="rIdX" ou r:link="rIdX"
            for rid in image_rids:
                if f'="{rid}"' in doc_xml or f'="{rid}"' in doc_xml:
                    return True

            return False

    except Exception as e:
        print(f"Erreur lors de l'inspection rapide du DOCX : {e}")
        return False


def convert_docx_to_pdf_with_libreoffice(docx_path: str) -> str:
    """Convertit un fichier DOCX en PDF via LibreOffice CLI."""
    output_dir = Path(docx_path).parent
    cmd = [
        "libreoffice",
        "--headless",
        "--convert-to",
        "pdf",
        docx_path,
        "--outdir",
        str(output_dir),
    ]
    subprocess.run(cmd, check=True)
    return str(output_dir / f"{Path(docx_path).stem}.pdf")


input_docx = "../example_data/example_1.docx"

# 1. Détection instantanée
needs_ocr = docx_body_has_images_fast(input_docx)

if needs_ocr:
    print("Image(s) détectée(s) dans le corps du DOCX : conversion PDF + VLM.")
    pdf_path = convert_docx_to_pdf_with_libreoffice(input_docx)

    engine_options = ApiVlmEngineOptions(
        runtime_type=VlmEngineType.API,
        url=f"{os.environ['LLM_API_URL'].rstrip('/')}/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['LLM_API_KEY']}"},
        params={
            "model": "gemma4-26b-moe",
            "temperature": 0.0,
        },
        timeout=120,
    )

    vlm_options = VlmConvertOptions.from_preset(
        "gemma_27b",
        engine_options=engine_options,
    )

    pipeline_options = VlmPipelineOptions(
        vlm_options=vlm_options,
        enable_remote_services=True,
    )

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_cls=VlmPipeline,
                pipeline_options=pipeline_options,
            )
        }
    )

    result = doc_converter.convert(pdf_path)

    if os.path.exists(pdf_path):
        os.remove(pdf_path)

else:
    print("Aucune image dans le corps : conversion directe DOCX via Docling.")
    doc_converter = DocumentConverter()
    result = doc_converter.convert(input_docx)

# 2. Export Markdown final
markdown_output = result.document.export_to_markdown()
print(markdown_output)

Aucune image dans le corps : conversion directe DOCX via Docling.


**PROCES VERBAL**

**ACCORD D’ENTREPRISE**

**NEGOCIATIONS ANNUELLES 2018**

Entre les soussignés :

**La société TRANSAVOIE, dont le siège social est sis : 926 avenue de la Houille Blanche – 73000 CHAMBERY, représentée par son Directeur,**

D’une part,

**Et :**

pour la CFDT

pour la CFE-CGC

pour la CFTC

pour la CGT,

pour FO,

D’autre part.

**PREAMBULE**

Des discussions ont été menées les 22 novembre, 13 et 20 décembre 2017, au titre de la négociation annuelle pour 2018.

Les éléments contextuels ont été abordés en préambule des négociations avec notamment :

En tenant compte de ce contexte, les intérêts et préoccupations exprimés par les Parties ont conduit à évoquer différentes hypothèses de travail, pour s’orienter de façon plus particulière vers la combinaison de différentes mesures :

- La réévaluation du taux horaire,
- La réévaluation d’éléments hors taux horaire,
- La création d’une Prime Qualité et de Non Accident.

Ce, en identifiant des leviers dans les accords et pra

# Repair docx

In [4]:
import zipfile
import shutil
import os

def repair_docx(input_docx: str, output_docx: str):
    """
    Nettoie une archive DOCX corrompue en supprimant les dossiers non conformes ([trash])
    et en recompressant proprement les fichiers XML.
    """
    # Répertoire temporaire pour extraire les fichiers
    extract_dir = input_docx + "_extracted"
    
    try:
        # Extraire l'archive
        with zipfile.ZipFile(input_docx, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        # Supprimer le dossier [trash] s'il existe
        trash_dir = os.path.join(extract_dir, "[trash]")
        if os.path.exists(trash_dir):
            shutil.rmtree(trash_dir)

        # Re-créer un ZIP propre
        with zipfile.ZipFile(output_docx, 'w', zipfile.ZIP_DEFLATED) as zip_out:
            for root, _, files in os.walk(extract_dir):
                for file in files:
                    full_path = os.path.join(root, file)
                    arcname = os.path.relpath(full_path, extract_dir)
                    zip_out.write(full_path, arcname)
                    
    finally:
        # Nettoyage du dossier d'extraction
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir)

repair_docx("../example_data/ACCOTEXT000036770370.docx", "../example_data/ACCOTEXT000036770370_clean.docx")